# Anomaly Detection Evaluation

Compare all three models:
- ROC/PR curves
- Per-accident-type detection rates
- Uncertainty calibration
- Detection latency analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from pathlib import Path

from src.data.dataset import NPPADWindowDataset
from src.models.anomaly import build_model
from src.evaluation.metrics import (
    compute_anomaly_scores, optimal_threshold, evaluate,
    per_accident_metrics, get_roc_curve, get_pr_curve
)
from src.evaluation.uncertainty import mc_dropout_scores
from src.utils.config import Config

sns.set_theme(style='whitegrid')

CHECKPOINT_DIR = Path('../checkpoints')
processed_dir = Path('../data/processed')

## 1. Load Models and Test Data

In [ ]:
test_dataset = NPPADWindowDataset(processed_dir / 'test.pt')
val_dataset = NPPADWindowDataset(processed_dir / 'val.pt')
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f'Test: {len(test_dataset)} windows')
print(f'Val: {len(val_dataset)} windows')

# Load trained models
model_names = ['autoencoder', 'lstm', 'transformer']
models = {}
for name in model_names:
    ckpt_path = CHECKPOINT_DIR / f'{name}_best.pt'
    if ckpt_path.exists():
        config = Config.from_yaml(f'../configs/{name}.yaml')
        model = build_model(config)
        ckpt = torch.load(ckpt_path, weights_only=True)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        models[name] = model
        print(f'Loaded {name} (val_loss={ckpt["val_loss"]:.6f}, epoch={ckpt["epoch"]})')
    else:
        print(f'No checkpoint for {name}')

## 2. ROC and PR Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

results = {}
for name, model in models.items():
    scores, labels, atypes = compute_anomaly_scores(model, test_loader)
    
    # Find optimal threshold on validation set
    val_scores, val_labels, _ = compute_anomaly_scores(model, val_loader)
    threshold = optimal_threshold(val_scores, val_labels)
    
    # Evaluate on test set
    metrics = evaluate(scores, labels, threshold)
    results[name] = {'scores': scores, 'labels': labels, 'atypes': atypes,
                     'threshold': threshold, 'metrics': metrics}
    
    # ROC
    fpr, tpr, _ = get_roc_curve(scores, labels)
    ax1.plot(fpr, tpr, label=f"{name} (AUROC={metrics.get('auroc', 0):.3f})")
    
    # PR
    prec, rec, _ = get_pr_curve(scores, labels)
    ax2.plot(rec, prec, label=f"{name} (AUPRC={metrics.get('auprc', 0):.3f})")

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend()

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Metrics Comparison Table

In [ ]:
import pandas as pd

metrics_table = pd.DataFrame({
    name: r['metrics'] for name, r in results.items()
}).T
metrics_table.index.name = 'Model'
print(metrics_table.to_string(float_format='%.4f'))

## 4. Per-Accident-Type Detection

In [ ]:
from data.scripts.preprocess import ACCIDENT_TYPES

inv_map = {v: k for k, v in ACCIDENT_TYPES.items()}

for name, r in results.items():
    print(f'\n=== {name.upper()} ===')
    per_type = per_accident_metrics(r['scores'], r['labels'], r['atypes'], r['threshold'])
    for atype_id, m in sorted(per_type.items()):
        atype_name = inv_map.get(atype_id, f'Type {atype_id}')
        det_rate = f"{m['detection_rate']:.1%}" if not np.isnan(m['detection_rate']) else 'N/A'
        print(f"  {atype_name:20s} | n={m['n_samples']:5d} | detection={det_rate} | mean_score={m['mean_score']:.4f}")

## 5. Uncertainty Calibration (MC Dropout)

In [ ]:
# MC Dropout uncertainty for each model
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5))
if len(models) == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, models.items()):
    mean_scores, std_scores = mc_dropout_scores(model, test_loader, n_samples=20)
    labels = results[name]['labels']
    
    normal_mask = labels == 0
    anomaly_mask = labels == 1
    
    ax.scatter(mean_scores[normal_mask], std_scores[normal_mask],
               alpha=0.3, s=10, label='Normal', color='green')
    ax.scatter(mean_scores[anomaly_mask], std_scores[anomaly_mask],
               alpha=0.3, s=10, label='Anomaly', color='red')
    ax.set_xlabel('Mean Anomaly Score')
    ax.set_ylabel('Score Std (Uncertainty)')
    ax.set_title(f'{name} - Uncertainty')
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Score Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5))
if len(models) == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    scores, labels = r['scores'], r['labels']
    ax.hist(scores[labels == 0], bins=50, alpha=0.6, label='Normal', color='green', density=True)
    ax.hist(scores[labels == 1], bins=50, alpha=0.6, label='Anomaly', color='red', density=True)
    ax.axvline(r['threshold'], color='black', linestyle='--', label=f"Threshold={r['threshold']:.4f}")
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Density')
    ax.set_title(f'{name}')
    ax.legend()

plt.suptitle('Anomaly Score Distributions', fontsize=14)
plt.tight_layout()
plt.show()